# SIH 2026 - Module 1 Hazard Intelligence - Annotated Walkthrough

**What this notebook does:** replays the exact Module 1 pipeline we built and ran, step by step,
with comments at every stage explaining *what* we did, *why*, and *where the data came from*.

**Pipeline:** Village boundaries -> DEM/slope/rainfall -> 4 hazard scores (flood, landslide, coastal, cloudburst)
-> multi-hazard fusion (AHP weights) -> Red Zone classification (GREEN/YELLOW/ORANGE/RED).

**Demo districts:** Chamoli (Uttarakhand) and Kendrapara (Odisha).

Run all cells in order. Needs `pip install -r backend/requirements.txt` once.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import geopandas as gpd

## 1. Village boundaries (the analysis unit)

We use **Survey of India / Census village polygons**. The raw shapefiles are in Lambert Conformal Conic (LCC_WGS84);
we filter to the demo district, rename columns to `village_id/name/district`, reproject to **EPSG:4326** (lon/lat)
and save as GeoJSON so the web map can read them.

This is exactly what `backend/data_pipeline/preprocess_villages.py` does.

In [ ]:
# --- raw file on disk, loaded for demonstration ---
SHAPEFILE = r"C:\Users\ROHAN\Downloads\indian_village_boundries\UTTARAKHAND.shp"
gdf_raw = gpd.read_file(SHAPEFILE)
print("Raw CRS:", gdf_raw.crs)
print("Total villages in state:", len(gdf_raw))

# Filter to CHAMOLI district only
chamoli = gdf_raw[gdf_raw["District"].astype(str).str.upper() == "CHAMOLI"].copy()
print("Chamoli villages:", len(chamoli))

# Rename to what the pipeline expects, reproject to GPS lon/lat (EPSG:4326)
chamoli = chamoli.rename(columns={"Vill_LGD": "village_id", "Vill_name": "name", "District": "district"})
chamoli = chamoli[["village_id", "name", "district", "geometry"]].to_crs("EPSG:4326")
chamoli.head(3)

In [ ]:
chamoli.plot(figsize=(11, 8), edgecolor="none", alpha=0.7)
plt_title = f"Chamoli district - {len(chamoli)} village polygons"
import matplotlib.pyplot as plt
plt.title(plt_title)

## 2. The 4 hazard engines

Each engine takes raster layers from Google Earth Engine, averages the values **inside each
village polygon** (`zonal_mean` = raster statistics per village), and returns a `0-1` score per village.

For this notebook we load the **already-downloaded real rasters** from `backend/data/raw/`
(they were exported from GEE). Database-IDs used on GEE:

| Hazard | GEE dataset | asset id |
|---|---|---|
| Flood | GFSM global flood susceptibility | `projects/floodsus/assets/fsm_ei5` |
| Landslide | ILSM landslide probability | `projects/ee-nirdeshsharmanith1/assets/ILSM_probability` |
| Elevation | SRTM 30m DEM | `USGS/SRTMGL1_003` |
| Rainfall | GPM IMERG monthly + CHIRPS daily | `NASA/GPM_L3/IMERG_MONTHLY_V07`, `UCSB-CHG/CHIRPS/DAILY` |
| Drainage | HydroSHEDS | `WWF/HydroSHEDS/03VFDEM` |
| Coastline | Natural Earth (local .geojson fallback) | (GEE NGDC/OSD unavailable) |

In [ ]:
RAW = Path("backend/data/raw")
PROCESSED = Path("backend/data/processed")
OUTPUT = Path("backend/data/output")
GEE_OUT = Path("backend/data/gee_output")

import rasterio

def raster_info(name):
    p = RAW / name
    if p.exists():
        with rasterio.open(p) as src:
            print(f"{name}: {src.width}x{src.height}, crs={src.crs}")
    else:
        print(f"{name}: MISSING (run the GEE pipeline first)")

for f in ["dem_chamoli.tif", "slope_chamoli.tif", "gpm_rainfall_chamoli.tif",
          "chirps_rainfall_chamoli.tif", "flood_gfsm_chamoli.tif", "ilsm_chamoli.tif"]:
    raster_info(f)

In [ ]:
from backend.hazards.base import zonal_mean, clamp01

def score_villages(villages, raster_path, transform=lambda v: clamp01(v)):
    out = villages.copy()
    out["_score"] = [
        transform(zonal_mean(raster_path, g)) if np.isfinite(zonal_mean(raster_path, g)) else 0.0
        for g in out.geometry
    ]
    return out

# FLOOD: GFSM class 1..5 -> 0..1  (class 5 = very high susceptibility)
flood = score_villages(chamoli, RAW / "flood_gfsm_chamoli.tif",
                       transform=lambda v: clamp01((v - 1) / 4))
print("Flood score per village (mean):", flood["_score"].mean().round(3))

# LANDSLIDE: ILSM probability is already 0..1
landslide = score_villages(flood, RAW / "ilsm_chamoli.tif")
print("Landslide score per village (mean):", landslide["_score"].mean().round(3))

## 3. Coastal + cloudburst scores

- **Coastal:** distance from village centroid to coastline; <=250 m -> 1.0; 250 m-3 km linear decay to 0.
  (Chamoli is inland so all = 0. For Kendrapara this is very important.)
- **Cloudburst:** weighted combination of DEM, slope, drainage, TWI, extreme rainfall.

In [ ]:
from backend.config.settings import RAW_COASTLINE_FILE

# COASTAL
coast = gpd.read_file(RAW_COASTLINE_FILE)
proj_villages = chamoli.to_crs(epsg=3857)   # metres
proj_coast = coast.to_crs(epsg=3857).geometry.unary_union
dists = [g.centroid.distance(proj_coast) for g in proj_villages.geometry]

def coastal_score(dist, hard=250.0, band=3000.0):
    return 1.0 if dist <= hard else float(max(0.0, 1.0 - (dist - hard) / (band - hard)))

coastal = [coastal_score(d) for d in dists]
print("Coastal erosion score (min/mean/max):",
      round(min(coastal), 3), round(np.mean(coastal), 3), round(max(coastal), 3))

In [ ]:
from backend.hazards.cloudburst import WEIGHTS, NORM

# CLOUDBURST: weighted sum of normalised layers
layers = {}
for key, fname in [("dem", "dem_chamoli.tif"), ("slope", "slope_chamoli.tif"),
                   ("drainage", None), ("twi", None), ("extreme_rain", None)]:
    if fname and (RAW / fname).exists():
        layers[key] = [zonal_mean(RAW / fname, g) for g in chamoli.geometry]
    else:
        layers[key] = [0.0] * len(chamoli)  # not available offline -> 0 contribution

cloud = sum(WEIGHTS[k] * clamp01(np.array(layers[k]) / NORM[k]) for k in WEIGHTS)
print("Cloudburst score per village (mean):", cloud.mean().round(3))

## 4. Combine into the final village table

Now every village has 4 hazard features. This is the **input to fusion**.

In [ ]:
df = chamoli.copy()
df["flood_score"] = flood["_score"]
df["landslide_score"] = landslide["_score"]
df["coastal_erosion_score"] = coastal
df["cloudburst_score"] = cloud
df["district"] = "Chamoli"
df["state"] = "Uttarakhand"
table = df[["village_id", "name", "district", "flood_score",
           "landslide_score", "coastal_erosion_score", "cloudburst_score"]]
table.round(3).head(10)

## 5. Multi-hazard fusion (AHP expert weights)

Weights derived from expert pairwise comparisons (sum = 1):
flood 0.35, landslide 0.30, coastal 0.20, cloudburst 0.15.

`multi_hazard = 0.35*flood + 0.30*landslide + 0.20*coastal + 0.15*cloudburst`

Red Zone rule:

| multi_hazard | Category | Zone |
|---|---|---|
| < 0.25 | Low | GREEN |
| 0.25-0.50 | Moderate | YELLOW |
| 0.50-0.75 | High | ORANGE |
| >= 0.75 | Very High | RED |

In [ ]:
from backend.fusion.red_zone import DEFAULT_WEIGHTS, CATEGORY_THRESHOLDS, CATEGORY_NAMES, RED_ZONE_RULE

W = DEFAULT_WEIGHTS
df["multi_hazard"] = (
    W["flood"] * df["flood_score"]
    + W["landslide"] * df["landslide_score"]
    + W["coastal"] * df["coastal_erosion_score"]
    + W["cloudburst"] * df["cloudburst_score"]
).clip(0, 1)

def category(score):
    return CATEGORY_NAMES[sum(score >= t for t in CATEGORY_THRESHOLDS)]

df["risk_category"] = df["multi_hazard"].apply(category)
df["red_zone_status"] = df["risk_category"].map(RED_ZONE_RULE)

df.groupby("red_zone_status")["village_id"].count()

## 6. Machine-learning variant (taught model)

Optional second method: train a model on the 4 features to predict *'this village got hit by
a disaster'*. With real NDRF records available, set `LABELS_CSV`; otherwise a documented
placeholder label (Very-High AHP score, or high flood AND high landslide) drives the training.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

FEATURES = ["flood_score", "landslide_score", "coastal_erosion_score", "cloudburst_score"]

# placeholder labels until real records are supplied
df["label"] = ((df["multi_hazard"] >= 0.55)
               | ((df["flood_score"] >= 0.55) & (df["landslide_score"] >= 0.55))).astype(int)

X, y = df[FEATURES].values, df["label"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
}
best, best_score = None, -1
for name, est in models.items():
    a = cross_val_score(est, X_tr, y_tr, cv=5, scoring="roc_auc").mean()
    print(f"{name}: CV ROC-AUC = {a:.3f}")
    if a > best_score:
        best, best_score = est, a

best.fit(X_tr, y_tr)
print("Test ROC-AUC:", round(roc_auc_score(y_te, best.predict_proba(X_te)[:, 1]), 3))
print(classification_report(y_te, best.predict(X_te), target_names=["safe", "hit"]))
if hasattr(best, "feature_importances_"):
    print("Feature importances:", dict(zip(FEATURES, best.feature_importances_.round(3))))

## 7. Save outputs (this is the Module 1 output form)

The final artifacts the dashboard consumes:
- `backend/data/processed/village_risk_<district>.csv`  - per village, 4 scores + multi_hazard + zone
- `backend/data/gee_output/village_risk_<district>.geojson` - map-ready polygons with the same attributes
- `backend/data/gee_output/*_score_<district>.tif` - hazard raster + multi_hazard + relocation priority
- `backend/data/output/fusion_model.joblib` - trained ML model (Module 2 ready)

**Output form = a per-village table + a colour-coded map layer (GREEN/YELLOW/ORANGE/RED)**
plus the relocation-priority ranking used by Modules 2 & 3.

In [ ]:
out_cols = ["village_id", "name", "district", *FEATURES, "multi_hazard", "risk_category", "red_zone_status"]
df_out = df[out_cols].sort_values("multi_hazard", ascending=False)
print(df_out.head(10).round(3).to_string(index=False))
# uncomment to actually write:
# df_out.to_csv(PROCESSED / "village_risk_chamoli.csv", index=False)
# df[out_cols + ["geometry"]].to_file(GEE_OUT / "village_risk_chamoli.geojson", driver="GeoJSON")

## Done - what Module 1 gives you

1. Every village in Chamoli/Kendrapara scored 0-1 on 4 hazards.
2. One combined multi-hazard score (AHP, explainable) + model risk (learned).
3. Red Zone class: GREEN / YELLOW / ORANGE / RED.
4. GeoJSON + CSV + rasters ready for the web dashboard.

Next: Module 2 exposures the red-zone villages with population & carrying capacity.